# Financial NLP Intelligence: Naive Bayes vs Logistic Regression vs SVM

**Apeiron AI — Boundless Possibilities, Infinite Potential**

## Objective
Build an end-to-end NLP pipeline using the FiQA-2018 financial dataset:
- Text preprocessing
- Tokenization
- TF-IDF representation
- Model comparison
- Evaluation
- Save best model for deployment

## 1. Setup & Imports

In [ ]:

import numpy as np
import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json

from datasets import load_dataset
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import *

nltk.download("punkt")
nltk.download("stopwords")


## 2. Load Dataset

In [ ]:

dataset=load_dataset("pauri32/fiqa-2018")

train_df=pd.DataFrame(dataset['train'])
val_df=pd.DataFrame(dataset['validation'])
test_df=pd.DataFrame(dataset['test'])

print(train_df.head())
print(train_df.columns)


## 3. Dataset Exploration

In [ ]:

print(train_df.shape)

train_df.head()


## 4. Text Preprocessing

In [ ]:

stemmer=PorterStemmer()
stop_words=set(stopwords.words('english'))

def preprocess(text):

    text=str(text).lower()
    text=re.sub(r'[^a-zA-Z ]','',text)

    words=word_tokenize(text)

    words=[
        stemmer.stem(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)


## 5. Apply Preprocessing

In [ ]:

TEXT_COL='sentence'

train_df["clean_text"]=train_df[TEXT_COL].apply(preprocess)
val_df["clean_text"]=val_df[TEXT_COL].apply(preprocess)
test_df["clean_text"]=test_df[TEXT_COL].apply(preprocess)


## 6. TF-IDF Representation

In [ ]:

vectorizer=TfidfVectorizer(max_features=5000)

X_train=vectorizer.fit_transform(train_df["clean_text"])
X_test=vectorizer.transform(test_df["clean_text"])

LABEL_COL='label'

y_train=train_df[LABEL_COL]
y_test=test_df[LABEL_COL]


## 7. Define Models

In [ ]:

models={

"NaiveBayes":MultinomialNB(),

"LogisticRegression":
LogisticRegression(max_iter=1000),

"SVM":
SVC(probability=True)

}


## 8. Train Models

In [ ]:

results=[]
trained_models={}

for name,model in models.items():

    model.fit(X_train,y_train)

    preds=model.predict(X_test)

    acc=accuracy_score(y_test,preds)
    precision=precision_score(y_test,preds,average='weighted')
    recall=recall_score(y_test,preds,average='weighted')
    f1=f1_score(y_test,preds,average='weighted')

    results.append(
        [name,acc,precision,recall,f1]
    )

    trained_models[name]=model


## 9. Results

In [ ]:

results_df=pd.DataFrame(
results,
columns=[
"Model",
"Accuracy",
"Precision",
"Recall",
"F1"
]
)

results_df


## 10. Confusion Matrix

In [ ]:

best_model_name=results_df.sort_values(
'F1',
ascending=False
).iloc[0]["Model"]

best_model=trained_models[best_model_name]

preds=best_model.predict(X_test)

cm=confusion_matrix(y_test,preds)

plt.figure(figsize=(8,5))
sns.heatmap(cm,annot=True,fmt='d')
plt.show()


## 11. Save Model

In [ ]:

pickle.dump(
best_model,
open("../model/best_model.pkl","wb")
)

pickle.dump(
vectorizer,
open("../model/tfidf_vectorizer.pkl","wb")
)

with open(
"../model/config.json",
"w"
) as f:

    json.dump({
        "model":best_model_name
    },f,indent=2)

print("Saved")


## 12. Summary

In [ ]:

print(results_df)
